# Prueba Tecnica - Paulo Hubert

## Strategy outline:

First I conducted some descriptivie analysis and visualization to understand data structure and availability. A few noteworthy points:

1. Data availability is highly different for different SKUs and regions
2. There are many zero-inflated series, with separated waves of non-zero values
3. There's no censored data due to lack of stock, as stocks never fall below 14%
4. Time span is less than a year, so seasonality analysis is not possible

Given the heterogeneity between the different SKUs and regions, and for optimization purposes, we will classify each time series in 3 categories:

1. Low data: series with 5 datapoints or less.
2. Zero-inflated: series with more than 5 datapoints where proportion of 0s is at least 40%
3. Full series: more than 5 datapoints and proportion of zeros lass than 40%

### Goal of this analysis

Define the infrastructure for a model selection engine that will select the best model for each time series (combination of INVENTORY_ID and REGION) and then generate forecasts based on the best model.
The structure should be flexbile, making the incorporation of new models easy, ideally automatically incorporating the new model into the backtest / model selection features.

### Candidate models

1. Naive (mean + standard deviation)
2. ARIMA
3. TSB

In [ ]:
import warnings

# Suppress all warnings globally in this notebook/session
warnings.filterwarnings("ignore")

%load_ext autoreload
%autoreload 2

# Data ingestion

In [4]:
import pandas as pd

df = pd.read_csv("data/Technical_test.df", decimal = ',')
df.head()

,INVENTORY_ID,REGION,DS,SALES,PERC_STOCK
0,SKU-1,FC-BA,2025-11-02,2,0.29
1,SKU-1,FC-PE,2025-11-02,2,0.29
2,SKU-1,FC-PE,2025-11-09,1,0.14
3,SKU-1,FC-PR,2025-11-02,3,0.43
4,SKU-1,FC-RJ,2025-10-26,1,0.14


# Descriptive analysis

In [3]:
df['INVENTORY_ID'].value_counts()

SKU-77    144
SKU-76    144
SKU-66    144
SKU-15    144
SKU-53    144
         ... 
SKU-68      1
SKU-34      1
SKU-40      1
SKU-61      1
SKU-64      1
Name: INVENTORY_ID, Length: 78, dtype: int64

In [4]:
# How many classes of data availability are there?
df['INVENTORY_ID'].value_counts().value_counts().sort_index(ascending = False)

144    29
143     2
140     1
135     4
132     2
24      1
22      1
20      2
18      2
16      1
15      2
14      1
12      6
11      2
10      1
9       4
8       1
7       1
6       3
5       3
2       4
1       5
Name: INVENTORY_ID, dtype: int64

In [5]:
# Time span
df['DS'].min(), df['DS'].max()

('2025-07-13', '2025-12-21')

In [6]:
# How many datapoints by series
dfg = df.groupby(['INVENTORY_ID', 'REGION']).agg({'DS' : "nunique"}).reset_index()
dfg.sort_values('DS', inplace = True)
dfg['DS'].value_counts()

24    190
2      50
1      49
3      30
22     22
23     15
5       3
21      3
20      2
7       1
14      1
10      1
12      1
Name: DS, dtype: int64

### Stock analysis: are there any censored datapoints?

In [7]:
df['PERC_STOCK'].describe()

count    5789.000000
mean        0.930301
std         0.183472
min         0.140000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000000
Name: PERC_STOCK, dtype: float64

# Time series classification

In [8]:
import sys
from pathlib import Path

lib_path = str(Path("anlytics-lib").resolve())
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)

from analytics_lib.classification import classify_time_series

classified_df = classify_time_series(df)

In [9]:
# Count the unique occurrences of each combination of INVENTORY_ID and REGION for each SERIES_CLASS
classified_df.groupby('SERIES_CLASS')[['INVENTORY_ID', 'REGION']].apply(lambda x: x.drop_duplicates().shape[0])

SERIES_CLASS
Full series      169
Low data         132
Zero-inflated     67
dtype: int64

# Model selection

In [13]:
import sys
from pathlib import Path

lib_path = str(Path("anlytics-lib").resolve())
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)

from analytics_lib.backtest_models import backtest_all_series

# Model selection parameters
th = 1
leave_out_k = 1
conf = 0.95

# Available models by series category
models_by_category = {
    "Low data": ["naive"],
    "Zero-inflated": ["naive", "tsb"],
    "Full series": ["naive", "arima", "tsb"],
}

### Backtesting each class separately

In [16]:

low_data_df = classified_df[classified_df["SERIES_CLASS"] == "Low data"]

# Right now is doesn't make sense to apply model selection to most any series in this class,
# as there is a single available model.
# Also many series in this class have a single datapoint, rendering model selection impossible
backtest_low_data = backtest_all_series(
    low_data_df,
    th=th,
    leave_out_k=leave_out_k,
    conf=conf,
    model_names=models_by_category["Low data"],
    n_jobs=-1,
)

In [17]:
zero_inflated_df = classified_df[classified_df["SERIES_CLASS"] == "Zero-inflated"]

backtest_zero_inflated = backtest_all_series(
    zero_inflated_df,
    th=th,
    leave_out_k=leave_out_k,
    conf=conf,
    model_names=models_by_category["Zero-inflated"],
    n_jobs=-1,
)

In [18]:
full_series_df = classified_df[classified_df["SERIES_CLASS"] == "Full series"]

backtest_full_series = backtest_all_series(
    zero_inflated_df,
    th=th,
    leave_out_k=leave_out_k,
    conf=conf,
    model_names=models_by_category["Full series"],
    n_jobs=-1,
)

/home/paulo/.pyenv/versions/3.10.13/envs/main-env/lib/python3.10/site-packages/pandas/core/indexes/multi.py:643: DeprecationWarning: `cumproduct` is deprecated as of NumPy 1.25.0, and will be removed in NumPy 2.0. Please use `cumprod` instead.
  codes = cartesian_product(codes)
/home/paulo/.pyenv/versions/3.10.13/envs/main-env/lib/python3.10/site-packages/pandas/core/reshape/util.py:60: DeprecationWarning: `product` is deprecated as of NumPy 1.25.0, and will be removed in NumPy 2.0. Please use `prod` instead.
  return [
/home/paulo/.pyenv/versions/3.10.13/envs/main-env/lib/python3.10/site-packages/pandas/core/reshape/util.py:60: DeprecationWarning: `product` is deprecated as of NumPy 1.25.0, and will be removed in NumPy 2.0. Please use `prod` instead.
  return [
/home/paulo/.pyenv/versions/3.10.13/envs/main-env/lib/python3.10/site-packages/pandas/core/indexes/multi.py:643: DeprecationWarning: `cumproduct` is deprecated as of NumPy 1.25.0, and will be removed in NumPy 2.0. Please use `c

### Flatten point results for reporting

In [19]:
def flatten_point_results(results_dict, series_class):
    rows = []
    for inventory_id, regions_dict in results_dict.items():
        for region, payload in regions_dict.items():
            if payload is None:
                continue
            point_results = payload.get("point_results", {})
            for model_name, metrics in point_results.items():
                rows.append(
                    {
                        "INVENTORY_ID": inventory_id,
                        "REGION": region,
                        "SERIES_CLASS": series_class,
                        "MODEL": model_name,
                        "MAE": metrics.get("MAE"),
                        "RMSE": metrics.get("RMSE"),
                        "MAPE": metrics.get("MAPE"),
                        "MASE": metrics.get("MASE"),
                    }
                )
    return rows

point_rows = []
point_rows += flatten_point_results(backtest_low_data, "Low data")
point_rows += flatten_point_results(backtest_zero_inflated, "Zero-inflated")
point_rows += flatten_point_results(backtest_full_series, "Full series")

final_point_metrics_df = pd.DataFrame(point_rows)
final_point_metrics_df.sort_values(
    ["INVENTORY_ID", "REGION", "SERIES_CLASS", "MODEL"], inplace=True
)
final_point_metrics_df.reset_index(drop=True, inplace=True)
final_point_metrics_df

,INVENTORY_ID,REGION,SERIES_CLASS,MODEL,MAE,RMSE,MAPE,MASE
0,SKU-1,FC-RJ,Low data,naive,0.5,0.5,50.000000,0.500000
1,SKU-1,FC-SP,Low data,naive,4.5,4.5,45.000000,0.642857
2,SKU-11,FC-SP,Full series,arima,NaN,NaN,NaN,NaN
3,SKU-11,FC-SP,Full series,naive,1.0,1.0,100.000000,NaN
4,SKU-11,FC-SP,Full series,tsb,1.0,1.0,100.000000,NaN
...,...,...,...,...,...,...,...,...
363,SKU-78,FC-BA,Low data,naive,4.5,4.5,450.000000,1.500000
364,SKU-78,FC-RJ,Low data,naive,8.5,8.5,850.000000,2.833333
365,SKU-78,FC-SP,Low data,naive,25.5,25.5,196.153846,2.318182
366,SKU-8,FC-SP,Low data,naive,0.5,0.5,50.000000,0.500000


# Best model selection

In [20]:
# Selection setup
# error_type: "point" or "interval"
error_type = "point"

# Point metrics: MAE, RMSE, MAPE, MASE
# Interval metric: CI_COVERAGE
selected_metric = "MAE"

# Direction of optimization for the selected metric:
# - point errors -> minimize
# - CI coverage -> maximize
optimize = "min" if error_type == "point" else "max"

In [21]:
def build_model_comparison_tables(result_objects):
    point_rows = []
    interval_rows = []

    for series_class, results_dict in result_objects.items():
        for inventory_id, regions_dict in results_dict.items():
            for region, payload in regions_dict.items():
                if payload is None:
                    continue

                for model_name, metrics in payload.get("point_results", {}).items():
                    point_rows.append(
                        {
                            "INVENTORY_ID": inventory_id,
                            "REGION": region,
                            "SERIES_CLASS": series_class,
                            "MODEL": model_name,
                            "MAE": metrics.get("MAE"),
                            "RMSE": metrics.get("RMSE"),
                            "MAPE": metrics.get("MAPE"),
                            "MASE": metrics.get("MASE"),
                        }
                    )

                for model_name, metrics in payload.get("interval_results", {}).items():
                    interval_rows.append(
                        {
                            "INVENTORY_ID": inventory_id,
                            "REGION": region,
                            "SERIES_CLASS": series_class,
                            "MODEL": model_name,
                            "CI_COVERAGE": metrics.get("CI_COVERAGE"),
                        }
                    )

    return pd.DataFrame(point_rows), pd.DataFrame(interval_rows)


result_objects = {
    "Low data": backtest_low_data,
    "Zero-inflated": backtest_zero_inflated,
    "Full series": backtest_full_series,
}

point_df, interval_df = build_model_comparison_tables(result_objects)

if error_type == "point":
    comparison_df = point_df.copy()
else:
    comparison_df = interval_df.copy()

metric_values = pd.to_numeric(comparison_df[selected_metric], errors="coerce")
valid_df = comparison_df.loc[metric_values.notna()].copy()
valid_df[selected_metric] = metric_values[metric_values.notna()]

if optimize == "min":
    best_idx = valid_df.groupby(["INVENTORY_ID", "REGION"])[selected_metric].idxmin()
else:
    best_idx = valid_df.groupby(["INVENTORY_ID", "REGION"])[selected_metric].idxmax()

best_model_by_series = (
    valid_df.loc[best_idx, ["INVENTORY_ID", "REGION", "SERIES_CLASS", "MODEL", selected_metric]]
    .rename(columns={"MODEL": "BEST_MODEL", selected_metric: "BEST_METRIC_VALUE"})
    .sort_values(["INVENTORY_ID", "REGION"])
    .reset_index(drop=True)
)

best_model_by_series

,INVENTORY_ID,REGION,SERIES_CLASS,BEST_MODEL,BEST_METRIC_VALUE
0,SKU-1,FC-RJ,Low data,naive,5.000000e-01
1,SKU-1,FC-SP,Low data,naive,4.500000e+00
2,SKU-11,FC-SP,Zero-inflated,naive,1.000000e+00
3,SKU-13,FC-SP,Zero-inflated,tsb,2.427030e+00
4,SKU-15,FC-BA,Full series,arima,6.161008e-31
...,...,...,...,...,...
95,SKU-78,FC-BA,Low data,naive,4.500000e+00
96,SKU-78,FC-RJ,Low data,naive,8.500000e+00
97,SKU-78,FC-SP,Low data,naive,2.550000e+01
98,SKU-8,FC-SP,Low data,naive,5.000000e-01


In [ ]:
from pathlib import Path

output_file = "results/best_model_by_series.csv"
best_model_by_series.to_csv(output_file, index=False)

Saved: outputs/best_model_by_series.csv


In [23]:
# Relative frequency of selected best model by series class
best_model_col = "BEST_MODEL" if "BEST_MODEL" in best_model_by_series.columns else "MODEL"

best_model_summary_df = (
    best_model_by_series.groupby("SERIES_CLASS")[best_model_col]
    .value_counts(normalize=True)
    .rename("RELATIVE_FREQUENCY")
    .mul(100)
    .reset_index()
    .sort_values(["SERIES_CLASS", "RELATIVE_FREQUENCY"], ascending=[True, False])
    .reset_index(drop=True)
)

best_model_summary_df

,SERIES_CLASS,BEST_MODEL,RELATIVE_FREQUENCY
0,Full series,arima,100.000000
1,Low data,naive,100.000000
2,Zero-inflated,tsb,53.658537
3,Zero-inflated,naive,46.341463


# Generating forecasts

In [24]:
import sys
from pathlib import Path

lib_path = str(Path("anlytics-lib").resolve())
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)

from analytics_lib.forecast_models import ForecastModels

# Load selected best model table if it is not already in memory
if "best_model_by_series" not in globals():
    best_model_path = Path("outputs") / "best_model_by_series.csv"
    best_model_by_series = pd.read_csv(best_model_path)

# Support either BEST_MODEL (preferred) or MODEL column names
model_col = "BEST_MODEL" if "BEST_MODEL" in best_model_by_series.columns else "MODEL"

forecast_horizon = 6
forecast_conf = conf if "conf" in globals() else 0.95

one_step_rows = []
all_series_6w_rows = []

for _, row in best_model_by_series.iterrows():
    inventory_id = row["INVENTORY_ID"]
    region = row["REGION"]
    model_name = row[model_col]

    series_df = df.loc[
        (df["INVENTORY_ID"] == inventory_id) & (df["REGION"] == region),
        ["DS", "SALES"],
    ].copy()

    if series_df.empty:
        continue

    try:
        # Fit on full history and forecast the next 6 weekly steps
        pred_6w = ForecastModels.predict(
            model_name,
            series_df,
            th=forecast_horizon,
            conf=forecast_conf,
        )
    except Exception:
        continue

    pred_6w = pred_6w.copy()
    pred_6w["INVENTORY_ID"] = inventory_id
    pred_6w["REGION"] = region
    pred_6w["MODEL"] = model_name
    all_series_6w_rows.append(pred_6w)

    # 1-step ahead prediction = first row of 6-step forecast
    first_step = pred_6w.iloc[0]
    one_step_rows.append(
        {
            "INVENTORY_ID": inventory_id,
            "REGION": region,
            "MODEL": model_name,
            "DS": first_step["DS"],
            "forecast": first_step["forecast"],
            "lower": first_step["lower"],
            "upper": first_step["upper"],
        }
    )

one_step_forecast_df = pd.DataFrame(one_step_rows)
one_step_forecast_df.sort_values(["INVENTORY_ID", "REGION"], inplace=True)
one_step_forecast_df.reset_index(drop=True, inplace=True)

series_6w_forecasts_df = pd.concat(all_series_6w_rows, ignore_index=True) if all_series_6w_rows else pd.DataFrame()

aggregate_6w_forecast_df = (
    series_6w_forecasts_df.groupby("DS", as_index=False)[["forecast", "lower", "upper"]]
    .sum()
    .sort_values("DS")
    .reset_index(drop=True)
)

# NOTE: Summing lower/upper bounds from individual series is a knowingly poor approximation
# for aggregated uncertainty because cross-series dependence is ignored.

/home/paulo/.pyenv/versions/3.10.13/envs/main-env/lib/python3.10/site-packages/pandas/core/dtypes/cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
/home/paulo/.pyenv/versions/3.10.13/envs/main-env/lib/python3.10/site-packages/pandas/core/dtypes/cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
/home/paulo/.pyenv/versions/3.10.13/envs/main-env/lib/python3.10/site-packages/pandas/core/dtypes/cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/r

In [25]:
one_step_forecast_df.to_csv('forecasts/one_step_forecast.df') # saving as .df to go around gitignore

OSError: Cannot save file into a non-existent directory: 'forecasts'

In [ ]:

aggregate_6w_forecast_df.to_csv('forecasts/aggregate_6w_forecast.df') # saving as .df to go around gitignore

In [28]:
aggregate_6w_forecast_df

,DS,forecast,lower,upper
0,2025-10-26,1.437500,0.727436,2.147564
1,2025-11-02,9.687500,4.631498,14.743502
2,2025-11-09,10.342890,4.631498,16.779988
3,2025-11-16,24.517493,5.661978,45.641999
4,2025-11-23,30.278458,9.131898,54.385038
5,2025-11-30,30.328458,9.131898,54.873300
6,2025-12-07,31.252526,8.413406,58.701960
7,2025-12-14,25.335859,4.509343,51.433251
8,2025-12-21,30.880469,4.509343,63.980183
9,2025-12-28,186060.587303,-491.078100,4387.142086
